# 06: Blocking with K=50 (recall at K=20/30/50) + miss analysis
Honest recall: the pools are the complete train sources (normalized by `work/norm_full.py`), not the small dev sample. Memory-safe: per (source, country), small query chunks, absolute document-frequency cap.

In [1]:
import os, sys, time, gc
os.environ['TMP']=os.environ['TEMP']='D:/tmp'
sys.path.insert(0,'D:/Amazon_ML_Challenge')
import pandas as pd, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from src.norm import norm_name, norm_addr
W='D:/Amazon_ML_Challenge/work/'; TR='D:/Amazon_ML_Challenge/Hackathon/Datasets/Train Datasets/'
NQ=5000
gt=pd.read_csv(TR+'train_ground_truth.tsv',sep='\t',dtype=str,keep_default_na=False,quoting=3)
gt=gt.sample(NQ,random_state=0)
need=set(gt.source1_entity_id)
q=pd.concat([c[c.entity_id.isin(need)] for c in pd.read_csv(TR+'train_source1.tsv',sep='\t',dtype=str,keep_default_na=False,quoting=3,chunksize=500000)]).reset_index(drop=True)
q['nn']=q.business_name.map(norm_name); q['na']=[norm_addr(a,c) for a,c in zip(q.business_address,q.country)]
g=gt.set_index('source1_entity_id').matched_entity_ids.str.split(',')
print(len(q), q.country.value_counts().to_dict())

5000 {'US': 2961, 'India': 2039}


In [2]:
def topk_cosine(Q,T,k,chunk=50):
    Tt=T.T.tocsr(); out=[]
    for i in range(0,Q.shape[0],chunk):
        S=(Q[i:i+chunk]@Tt).tocsr()
        for r in range(S.shape[0]):
            a,b=S.indptr[r],S.indptr[r+1]; ix=S.indices[a:b]; v=S.data[a:b]
            if len(v)>k: p=np.argpartition(-v,k)[:k]; ix,v=ix[p],v[p]
            out.append((ix,v))
    return out

K=50; MAXDF=20000
def run(src):
    """returns list (per query) of dict channel -> (entity_ids, scores)"""
    res=[{} for _ in range(len(q))]
    for c in q.country.unique():
        qi=np.flatnonzero(q.country.values==c)
        pool=pd.read_parquet(W+f'full/{src}_{c}.parquet'); eid=pool.entity_id.values
        for ch,col in (('name','nn'),('addr','na')):
            vec=TfidfVectorizer(token_pattern=r"\S+",lowercase=False,sublinear_tf=True,max_df=MAXDF,dtype=np.float32)
            T=vec.fit_transform(pool[col].values); Q=vec.transform(q[col].values[qi])
            for j,(ix,v) in zip(qi,topk_cosine(Q,T,K)): res[j][ch]=(eid[ix],v)
            del T,Q,vec; gc.collect()
        print(src,c,len(pool),'rows done',round(time.time()-t0),'s',flush=True); del pool; gc.collect()
    return res
t0=time.time(); R={s:run(s) for s in ('s2','s3')}

s2 India 2017799 rows done 95 s


s2 US 3016817 rows done 304 s


s3 India 2115547 rows done 419 s


s3 US 3170056 rows done 574 s


## Recall at K=20/30/50 (top-K by cosine within each channel)

In [3]:
def topn(ent,sc,k):
    o=np.argsort(-sc)[:k]; return set(ent[o])
def evaluate(src,k):
    rows=[]
    for i,qid in enumerate(q.entity_id):
        truth={t for t in g[qid] if t.startswith(src.upper())}
        if not truth: continue
        n=topn(*R[src][i]['name'],k); a=topn(*R[src][i]['addr'],k)
        rows.append((q.country[i],len(truth),len(truth&(n|a)),len(n|a)))
    d=pd.DataFrame(rows,columns=['country','truth','hit','ncand']); t=d.groupby('country').agg(truth=('truth','sum'),hit=('hit','sum'),cands=('ncand','mean'))
    t['recall']=(t.hit/t.truth).round(3); all_=d[['truth','hit']].sum()
    return round(all_.hit/all_.truth,3),round(d.ncand.mean(),1),t.recall.to_dict()
for s in ('s2','s3'):
    for k in (20,30,50): print(s,'K=',k,evaluate(s,k))

s2 K= 20 (np.float64(0.948), np.float64(38.5), {'India': 0.937, 'US': 0.956})


s2 K= 30 (np.float64(0.958), np.float64(58.2), {'India': 0.946, 'US': 0.966})


s2 K= 50 (np.float64(0.965), np.float64(97.4), {'India': 0.953, 'US': 0.973})


s3 K= 20 (np.float64(0.932), np.float64(38.5), {'India': 0.898, 'US': 0.955})


s3 K= 30 (np.float64(0.94), np.float64(58.1), {'India': 0.909, 'US': 0.963})


s3 K= 50 (np.float64(0.952), np.float64(97.3), {'India': 0.926, 'US': 0.969})


## What is still missed at K=50?

In [4]:
import re
miss=[]
for s in ('s2','s3'):
    for i,qid in enumerate(q.entity_id):
        got=topn(*R[s][i]['name'],50)|topn(*R[s][i]['addr'],50)
        for t in g[qid]:
            if t.startswith(s.upper()) and t not in got: miss.append((qid,s,t,i))
miss=pd.DataFrame(miss,columns=['s1','src','tid','i']); print(len(miss),'missed pairs'); need_ids=set(miss.tid)
rw=pd.concat([c[c.entity_id.isin(need_ids)] for s in ('2','3') for c in pd.read_csv(TR+f'train_source{s}.tsv',sep='	',dtype=str,keep_default_na=False,quoting=3,chunksize=500000)]).set_index('entity_id')
qq=q.set_index('entity_id')
miss['a_name']=qq.loc[miss.s1].business_name.values; miss['a_addr']=qq.loc[miss.s1].business_address.values; miss['country']=qq.loc[miss.s1].country.values
miss['b_name']=rw.loc[miss.tid].business_name.values; miss['b_addr']=rw.loc[miss.tid].business_address.values
def cat(r):
    b=re.sub(r"[\W\d_]","",r.b_name)
    if b and sum(ord(c)>127 for c in b)/len(b)>0.5: return 'native_script_name'
    if re.search(r"\.(com|in|net|org)|^@",r.b_name,re.I): return 'domain_style'
    from rapidfuzz import fuzz
    return 'name_similar' if fuzz.token_set_ratio(norm_name(r.a_name),norm_name(r.b_name))>=70 else 'name_different'
miss['cat']=miss.apply(cat,axis=1)
print(miss.groupby(['country','src','cat']).size().unstack('cat').fillna(0).astype(int))
pd.set_option('display.max_colwidth',60); pd.set_option('display.width',250)
for c_,x in miss.groupby('cat'): print('==',c_,len(x)); print(x.sample(min(6,len(x)),random_state=0)[['a_name','b_name','a_addr','b_addr']].to_string(index=False))

716 missed pairs


cat          domain_style  name_different  name_similar  native_script_name
country src                                                                
India   s2              0              19            42                 100
        s3              3              38           141                  84
US      s2              0              13           119                   0
        s3              1              21           135                   0
== domain_style 4
                             a_name            b_name                                                                                    a_addr                                              b_addr
Ecospace Apparels (India) Pvt. Ltd. @Ecospaceapparels         Hno748/1, Kalyan, 2, Gala1, Gr Flr16, Arihant Comp Purna Vill, Thane, Maharashtra                  Hno48/1, Kalyan, Thane, महाराष्ट्र
                 Olanyx Crest Group      @olanyxcrest                                                 725 24th Street, Unit 407, Washi

In [5]:
rows=[]
for s in ('s2','s3'):
    for i,qid in enumerate(q.entity_id):
        for ch,(e,v) in R[s][i].items(): rows+= [(qid,s,ch,x,float(y)) for x,y in zip(e,v)]
pd.DataFrame(rows,columns=['s1_id','src','channel','cand_id','cos']).to_parquet(W+'cand_full_k50.parquet'); print(len(rows))

987321
